In [ ]:
%matplotlib widget

# Leiden-Argentina-Bonn neutral hydrogen survey

THe Leiden-Argentina-Bonn survey was an all-sky survey of neutral hydrogen. Although the survey is superseded by more modern ones, what makes it useful for our purpose is the fact that it published brightness-temperature maps instead of directly converting to column-densities. A desciption with relevant references can be found here: https://lambda.gsfc.nasa.gov/product/foreground/fg_LAB_HI_Survey_info.html and the maps themselves can be downloaded here: https://lambda.gsfc.nasa.gov/data/foregrounds/HI/lab_healpix.tar.bz2

Download and unpack the maps. Modify the following parameter to refer to the location where YOU stored the fits files:

In [ ]:
ROOT_DIR = 'LAB-survey/LAB_Healpix/'

## Reading and accessing the data

The following cell contains code to read the files in your ROOT_DIR, read velocities, and convert them to frequencies. Carefully read through the following cell and execute it.

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
from astropy.table import QTable
import astropy.constants as constants
import healpy as hp
import scipy


def velocity_from_filename(file_name):
    r'''
    Returns and int containing the mean velocity of a LAB Healpix file in km/s. 
    Note that the number in the file name itself is the lower limit of velocity, 
    and the full range in a file is 10 km/s.

    Parameters
    ----------

    file_name : string: filename to obtain velocity from.
    '''
    # add 5 km/s to indicate the mean velocity, not the lower limit.
    return int(file_name.split('_cut')[-1][:-5]) + 5


def read_file(file_name):
    r'''
    Read a HEALPix map frame from the LAB survey.

    Parameters
    ----------

    file_name : string: filename to read.

    
    Returns
    -------

    tuple (astropy quantity, astropy quantity array) representing (mean_velocity, HEALPix_image) in 
    km/s and K respectively.
    '''
    velocity=velocity_from_filename(file_name)*u.km/u.s
    return (velocity, hp.read_map(file_name))


#def gaussian_beam_fn(fwhm=70*u.deg):
#    def fn(angular_distance):
#        d = angular_distance    
#        return np.exp(-(d.to(u.rad).value**2/(2*(fwhm.to(u.rad).value/2.355)**2)))
#    return fn


def airy_beam_fn(r_to_first_null=70*u.deg):
    r'''
    Returns a function(angular_distance) that returns the power beam gain of the airy pattern
    with peak-to-first-null given as a parameter to THIS function size at a certain angular distance.
    
    '''
    r_null = r_to_first_null.to(u.rad)/2
    def fn(angular_distance):
        d = angular_distance
        l = d.to(u.rad)*1.9158715
        return (2*scipy.special.j1(l/r_null)/(l/r_null))**2
    return fn






class LABSurvey:
    def __init__(self, root_dir):
        r'''
        Plot average spectra over parts of the sky. Survey can be downloaded from:
            https://lambda.gsfc.nasa.gov/data/foregrounds/HI/lab_healpix.tar.bz2
        
        Parameters
        ----------
        root_dir: string: path containing the HEALPIX FITS files of the LAB survey.

        Variables
        ---------
        
        images : 2D array of astropy quantities: the actual HEALPix maps. Indices: [frame, pixel]
        velocity : Array of astropy quantities, length number of frames, contains mean velocity
                   of each frame in km/s
        frequency : Array of astropy quantities, length number of frames, contains mean frequency 
                    of each frame in MHz.
        
        '''
        file_names = glob.glob(os.path.join(root_dir)+'*cut*.fits')
        self.nside = 512
        images = [read_file(name) for name in file_names]
        images.sort(key=lambda x: x[0])
        self.velocity = np.array([x[0].value for x in images])*u.km/u.s
        f0 = 1420.405751768*u.MHz
        self.frequency = (f0 - f0*self.velocity/constants.c).to(u.MHz)
        self.images = np.array([x[1] for x in images])*u.K


    def plot_map(self, map_index, **args):
        return hp.mollview(map=self.images[map_index], unit='K', title="Velocity: %s" % (self.velocity[map_index],), **args)


    def plot_beam_map(self, map_index, lon, lat, beam_size=70*u.deg, **args):
        beam_fn = airy_beam_fn(beam_size)
        vec = hp.ang2vec(theta=lon.to(u.deg).value, phi=lat.to(u.deg).value,
                         lonlat=True)
        mask = hp.query_disc(nside=self.nside, vec=vec, 
                             radius=np.pi)
        angular_distances = hp.rotator.angdist(vec, hp.pix2vec(nside=self.nside, ipix=mask))*u.rad
        beam = np.squeeze(beam_fn(angular_distances))
        image = self.images[map_index].copy()
        image[mask] *= beam
        return hp.mollview(map=image, unit='K', title="Velocity: %s" % (self.velocity[map_index],), **args)

    
    def spectrum(self, lon, lat, beam_fn=airy_beam_fn(70*u.deg), radius=180*u.deg):
        r'''
        '''
        vec = hp.ang2vec(theta=lon.to(u.deg).value, phi=lat.to(u.deg).value,
                         lonlat=True)
        mask = hp.query_disc(nside=self.nside, vec=vec, 
                             radius=min(radius.to(u.rad).value, np.pi))
        angular_distances = hp.rotator.angdist(vec, hp.pix2vec(nside=self.nside, ipix=mask))*u.rad
        beam = np.squeeze(beam_fn(angular_distances))
        return (self.images[:,mask]*beam[np.newaxis,:]).mean(axis=1)/beam.mean()

    
    def plot_spectrum(self, ax, lon, lat, beam_size=70*u.deg, beam_fn=airy_beam_fn, x_axis='velocity'):
        r'''
        Parameters
        ----------
        ax : Matplotlib Axis instance to use for plotting.
        lon : Astropy angular quantity: galactic longitude l of pointing centre, e.g. 85*u.deg.
        lat : Astropy angular quantity: galactic latitude b of pointing centre, e.g. 85*u.deg.
        beam_size : Astropy angular quantity: approximate FWHM of beam, assuming a Gaussian beam shape
                    or peak-to-first-null when using the airy pattern beam.
        beam_fn : The beam shape function. Either gaussian_beam_fn or airy_beam_fn.
        x_axis : String: 'velocity' or 'frequency'
        '''
        avg_spectrum = self.spectrum(lon=lon, lat=lat,
                                beam_fn=beam_fn(beam_size),
                                radius=3*beam_size)
        if x_axis=='velocity':
            ax.plot(self.velocity, avg_spectrum)
            ax.set_xlabel('Radio radial velocity [km/s]')
        elif x_axis=='frequency':
            ax.plot(self.frequency, avg_spectrum)
            ax.set_xlabel('Frequency [MHz]')
        else:
            raise ValueError("x_axis must be 'velocity' or 'frequency', not '%s'" % x_axis)
        ax.grid()
        
        ax.set_ylabel('Brightness temperature [K]')
        ax.set_title(r'Neutral Hydrogen $T_\mathrm{B}$ at $l,b$ = %.2f$^\circ$,%.2f$^\circ$; size %.2f$^\circ$ ' % 
                     (lon.to(u.deg).value, lat.to(u.deg).value, beam_size.to(u.deg).value))

# Instantiate the object that contains the entire survey.
LAB = LABSurvey(ROOT_DIR)

## All sky maps

The LAB object now contains the image frames, velocities, and frequencies. Let's print a quick table to see how frequencies and velocities map to the first index of the `LAB.images` array. We'll use the `astropy.table` module for that:


In [ ]:
frame_units = QTable(data=[np.arange(LAB.velocity.shape[0]),LAB.velocity, LAB.frequency],
                    names=['Index', 'Velocity', 'Frequency'])

print('\n'.join(frame_units.pformat(max_lines=-1)))

Let's now have a look at some of the maps. Read the `LABSurvey.plot_map()` method. Modify and execute the following line to show the maps at -105, -55, +5, +55, and +105 km/s. Play around with the max brightness as well.

In [ ]:
LAB.plot_map(45, min=0, max=100)

Let's now have a look at the effect of the antenna beam. Read the `LABSurvey.plot_beam_map()` method. Play with the following cell to explore what the antenna actually "sees". Try the beam size of a 25 m dish, a 3 m dish, a 70 cm dish, and a paint can. Use whatever frame and direction you like. Try different longitudes, latitudes, and max brightnesses.

In [ ]:
LAB.plot_beam_map(45, lon=75*u.deg, lat=0*u.deg, beam_size=20*u.deg, min=0, max=100)

## Spectra

The spectrum we observe is the weighted mean of the sky multiplied by the beam's power gain as a function of frequency. The function `LABSurvey.plot_spectrum` does that for you. Modify the numbers below to reflect the most likely part of the Milky way you'll observe next week, as well as the beam size you expect for the paint can. **Use the result to guide your sensitivity calculations and proposed channel band width.**

In [ ]:
fig, (ax_v, ax_f) = plt.subplots(2,1,figsize=(6,6),dpi=200)

lon=175*u.deg
lat=31.5*u.deg
beam_size=1*u.deg

LAB.plot_spectrum(ax_v,lon=lon, lat=lat, beam_size=beam_size, beam_fn=airy_beam_fn,
                  x_axis='velocity')

LAB.plot_spectrum(ax_f,lon=lon, lat=lat, beam_size=beam_size, beam_fn=airy_beam_fn,
                  x_axis='frequency')

fig.subplots_adjust(hspace=0.4)